<a href="https://colab.research.google.com/github/sreent/machine-learning/blob/main/Lectures/2%20K-Nearest%20Neighbours%20Classification/Guided%20Lab%3A%20K%E2%80%91Nearest%20Neighbours%20(KNN)%20Classification_%E2%80%94_Guided_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K‑Nearest Neighbours (KNN) Classification — Guided Lab 🔍

Inspired by: the step‑wise, task‑oriented DataCamp guided projects and the conceptual walk‑through in the lecture slides K‑Nearest Neighbours Classification.

Goal: Use scikit‑learn to train, tune and interpret a KNN classifier.  Rather than writing the algorithm from scratch, you’ll fill in the blanks in pre‑written cells.

In this lab, you’ll explore:
1.	How to split your dataset into training, validation, and test sets.
2.	The importance of scaling features when using distance‑based algorithms like KNN.
3.	How to choose the optimal number of neighbors (k) for the best bias–variance trade‑off.
4.	How to evaluate model performance using accuracy scores and confusion matrices.
5.	(Optional) How different distance metrics (Euclidean vs. Manhattan) might affect performance.

## 🛠️ Lab Instructions
1.	Look for cells marked # TODO. Replace each ... with working code (e.g., a function call, a Python object, etc.).
2.	Run the cells in order. It’s often helpful to run them multiple times to iterate.
3.	A solution key is provided after each part—keep it collapsed during your first attempt to avoid spoilers.

## 1. Load Packages & Set Seed

In the code block below, we import the core libraries: NumPy, pandas, scikit-learn modules (for model building, splitting, scaling), and data visualization libraries (Matplotlib, Seaborn). We also set a seed (random state) to ensure that your splits and any randomized processes are reproducible.

In [ ]:
# Imports 🚀  (Keep ONLY the ones you use.)
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine   # default dataset for speed – change if you wish
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

Why set a seed?

When you split data randomly or perform any randomized steps (e.g., shuffling), setting the random seed ensures that each time you run the code, you’ll get the same split, enabling consistent results and easier debugging.

## 2. Prepare Data 📦

Here, we load the Wine dataset from scikit‑learn. This is a small dataset with 13 numeric features that describe the chemical composition of various wines, labeled as one of three classes. (In practice, you might replace it with any dataset of your choice—e.g., a text dataset or an image dataset.)

### 2.1 Create train/validation/test splits

Why do a three-way split?
-	Training set: used to fit/train the model parameters.
-	Validation set: used to evaluate different hyperparameters (like k) and avoid overfitting.
-	Test set: used at the very end to get an unbiased performance estimate.

We aim for a 60/20/20 ratio. To achieve this:
1.	Split the entire dataset into train and a temporary set (X_temp, y_temp) with 60/40.
2.	Then split that temporary set in half (i.e., 20/20 overall) into validation and test.

In [ ]:
# Load the Wine dataset (13 numeric features, 3 classes)
X, y = load_wine(return_X_y=True)

# TODO ❶: Split into train, validation and test in the recommended 60/20/20 ratio
# hint: first do train vs temp, then temp → validation & test (slides 16‑17)
X_train, X_temp, y_train, y_temp = train_test_split(...)
X_val, X_test, y_val, y_test = train_test_split(...)

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)

Tip: For stratified splits, you can specify stratify=y to ensure that each split keeps the same proportion of classes—important for classification tasks with imbalanced classes.

### 2.2 Standardise Features ⚖️

Why scale?

KNN works by measuring distances between points. Features on larger scales can dominate distance computations, which can skew results. Scaling puts all features on a comparable range (often 𝜇=0, 𝜎=1 using StandardScaler), improving model performance and stability.

Important: You always .fit() the scaler on training data only. Then you .transform() all splits (train, validation, test) using those training scale parameters. That way, your model is not “peeking” at any future data.

In [ ]:
# TODO ❷: Fit StandardScaler **only on the training data**, then transform all three splits
scaler = StandardScaler()
scaler.fit(...)
X_train_s = ...
X_val_s   = ...
X_test_s  = ...

### 3. Baseline KNN Model 🏁

Here, let’s instantiate a basic KNN classifier with a fixed k=5 (i.e., 5 neighbors) and use 'euclidean' distance. Then we evaluate on the validation set. This is our initial baseline before we try tuning.

In [ ]:
# TODO ❸: Instantiate KNeighborsClassifier with k=5, metric='euclidean' (slide 18)
knn = KNeighborsClassifier(...)
knn.fit(X_train_s, y_train)

# Evaluate on validation set
val_pred = knn.predict(X_val_s)
val_acc  = accuracy_score(y_val, val_pred)
print(f"Validation accuracy: {val_acc:.3f}")

Interpretation: The validation accuracy tells us roughly how good our model is at classifying unseen data after training. But we can often do better by tuning hyperparameters like k.

## 4. Tune k to Balance Bias–Variance 🎚️

KNN’s number of neighbors (k) is one of its most critical hyperparameters:
-	Too small k ⇒ can overfit, capturing noise and having high variance.
-	Too large k ⇒ can underfit, oversimplifying the decision boundary and having high bias.

We’ll:
1.	Loop over different values of k from 1 to 30.
2.	For each k, train a KNN model on the training set.
3.	Record train accuracy and validation accuracy.
4.	Plot these curves and look for the sweet spot that maximizes validation accuracy.

In [ ]:
# TODO ❹: Sweep k from 1 → 30; store train & val accuracies for a plot
k_range      = range(1, 31)
train_scores = []
val_scores   = []

for k in k_range:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_s, y_train)
    train_scores.append(...)
    val_scores.append(...)

# Plot
plt.figure(figsize=(8,4))
plt.plot(k_range, train_scores, label="Train")
plt.plot(k_range, val_scores, label="Validation")
plt.axvline(x=k_range[np.argmax(val_scores)], color="black", ls="--", label="Best k")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Bias–Variance Trade‑off vs k (scaled features)")
plt.show()

Plot Interpretation:
-	When train accuracy is much higher than val accuracy, you might be overfitting.
-	Watch how val accuracy changes with k. The peak indicates the best trade‑off.

## 5. Final Model & Confusion Matrix 🧮

After finding the best_k on the validation set, we fix that hyperparameter and retrain on the combined training data (still only X_train_s, y_train—the combination means same split, but we now finalize that choice of hyperparameter). Finally, we check performance on the test set, which we haven’t used at all in the tuning process.

In [ ]:
best_k = k_range[np.argmax(val_scores)]
print("Best k from validation:", best_k)

final_knn = KNeighborsClassifier(n_neighbors=best_k)
final_knn.fit(X_train_s, y_train)

test_pred = final_knn.predict(X_test_s)
print("Test accuracy:", accuracy_score(y_test, test_pred))

cm = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(cm, display_labels=load_wine().target_names).plot(cmap="Blues")
plt.title("KNN Confusion Matrix (test set)")
plt.show()

Interpreting the Confusion Matrix
-	Diagonal entries (top-left to bottom-right) show correct classifications.
-	Off-diagonal entries show misclassifications, e.g., how often a class is mistaken for another.

For a 3-class dataset like Wine, each row corresponds to the actual class, and each column corresponds to the predicted class. If you see a large off-diagonal value, that means the model frequently confuses one class with another.

## 6. (Optional) Distance Metric Experiment 🔄

Why experiment with distance metrics?
	•	Euclidean (L2) is the most common.
	•	Manhattan (L1) can be more robust in some cases with outliers or certain data distributions.

Try repeating the k sweep with metric='manhattan' instead of the default 'euclidean' to see if your best k or your best accuracy changes.

In [ ]:
# TODO ❺: Your experiment here (copy‑paste the sweep loop & change the metric)


Compare the resulting plots or final accuracies. Sometimes, the difference is small, but it can be important in certain high‑dimensional or irregularly distributed data sets.

## 7. Reflection 📝
1.	Where did the biggest gains come from—scaling, tuning k, or changing metrics?
    -	Typically, feature scaling is crucial in distance-based algorithms to ensure that no single feature dominates.
	  -	Tuning k often has a more direct impact on the bias–variance trade-off.
	  -	Changing metrics might only help for certain data distributions, but it’s a valuable experiment.
2.	How would you adapt this workflow to a high‑dimensional text classification task?
	  -	You would still do a train/validation/test split.
	  -	Preprocessing might involve vectorizing text into numeric features (e.g., TF–IDF).
	  -	Scaling may or may not be applicable depending on your text representation.
	  -	You’d still tune k (and possibly other hyperparameters).
	  -	You might also look at dimensionality reduction or other transformations if you have thousands of features.

## 🏆 Final Thoughts

By following the steps above, you have:
1.	Loaded a dataset and split it properly.
2.	Scaled your features to ensure fair distance calculations.
3.	Trained a baseline KNN model, then tuned k to find a better bias–variance trade-off.
4.	Evaluated your best model using a test set and a confusion matrix.